# Intent Classification for Banking Chatbots (Topic #7 — `BANKING77`)

**Course:** NLP, Language Models & Analytics  
**Objective:** Classify customer support utterances into **77 fine-grained banking intents** while respecting real-world deployment constraints: **sub-50 ms CPU latency**, **subtle intent disambiguation**, and **confidence-gated human escalation** for ambiguous or out-of-scope queries.

### Architecture & Model Progression
1. **Track A (Classical Baseline):** Domain-aware preprocessing (preserving negation + normalizing currency/numbers) -> `TF-IDF (1–2 n-grams)` + `Logistic Regression`.
2. **Track B (Primary Transformer):** Minimal Unicode/whitespace cleaning -> Fine-tuned `distilbert-base-uncased`.
3. **Real-World Guardrail:** Validation-calibrated confidence threshold (`tau`) routing low-confidence queries to a human specialist.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import (
    BASELINE_MODEL_PATH,
    OUTPUTS_DIR,
    SEED,
    TRANSFORMER_BASE_MODEL,
    TRANSFORMER_MODEL_DIR,
)
from src.data import get_oos_and_edge_probes, load_banking77_splits
from src.preprocessing import (
    clean_classical_aggressive,
    clean_classical_domain_aware,
    clean_transformer_minimal,
)
from src.baseline import (
    load_baseline,
    predict_baseline_probs,
    run_preprocessing_ablation,
    tune_and_train_baseline,
)
from src.transformer_model import (
    get_device,
    load_transformer,
    predict_transformer_batch,
    set_global_seed,
    train_distilbert,
    transformer_weights_exist,
)
from src.evaluation import (
    benchmark_single_query_latency,
    compute_classification_metrics,
    evaluate_confidence_thresholds,
    export_metrics_tables,
    extract_error_cases,
    get_path_size_mb,
    get_per_class_f1_summary,
    get_top_confused_pairs,
    plot_class_distribution,
    plot_confidence_coverage_curve,
    plot_focused_confusion_cluster,
    plot_model_comparison,
    plot_top_confused_pairs,
    select_recommended_threshold,
)
from src.inference import IntentClassifierService

set_global_seed(SEED)
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

# Set to True only if you want to re-run Transformer fine-tuning inside this notebook.
FORCE_RETRAIN_TRANSFORMER = False

print(f'Project root: {PROJECT_ROOT}')
print(f'Active torch device: {get_device()}')
print(f'Transformer weights present: {transformer_weights_exist(TRANSFORMER_MODEL_DIR)}')

Project root: C:\Users\easha\OneDrive\Documents\SP JAIN\SEM 5\NLP\INDV\intent_classifier
Active torch device: cpu
Transformer weights present: True


## 2. Dataset Exploration & Stratified Split Verification
We load `mteb/banking77` (77 intents) and carve a **15% stratified validation split** (`SEED = 42`) from the official training set. Hyperparameters and confidence thresholds are tuned strictly on validation; the 3,076 test examples remain untouched until final evaluation.

In [2]:
bundle = load_banking77_splits(use_local_cache=True)
word_counts = pd.Series([len(t.split()) for t in bundle.train_texts])

print(f'Dataset source: {bundle.dataset_source}')
print(f'Classes: {len(bundle.label_names)}')
print(f'Splits -> Train: {len(bundle.train_texts)}, Val: {len(bundle.val_texts)}, Test: {len(bundle.test_texts)}')
print('Train query word-length summary:')
print(word_counts.describe(percentiles=[0.5, 0.9, 0.99]).round(2))

plot_class_distribution(bundle.train_labels, bundle.label_names, OUTPUTS_DIR / 'class_distribution.png')

Dataset source: mteb/banking77
Classes: 77
Splits -> Train: 8494, Val: 1499, Test: 3076
Train query word-length summary:
count    8494.00
mean       11.93
std         7.80
min         3.00
50%        10.00
90%        21.00
99%        42.00
max        79.00
dtype: float64


## 3. Dual-Track Preprocessing & Quantitative Ablation
Classical models need normalized numeric/currency tokens while **preserving negation words** (`not`, `no`, `never`, `isn't`, `can't`). Removing standard English stopwords blindly destroys polarity cues.

In [3]:
sample_examples = [
    "My contactless tap isn't going through for $49.99 even though I entered PIN 1234!",
    "I have NOT received my £200 refund after 5 days.",
]
demo_rows = [
    {
        'raw': s,
        'track_a_domain_aware': clean_classical_domain_aware(s),
        'ablation_aggressive_nltk': clean_classical_aggressive(s),
        'track_b_transformer': clean_transformer_minimal(s),
    }
    for s in sample_examples
]
display(pd.DataFrame(demo_rows))

ablation_df = run_preprocessing_ablation(bundle)
ablation_df.to_csv(OUTPUTS_DIR / 'preprocessing_ablation.csv', index=False)
display(ablation_df.round(4))

,raw,track_a_domain_aware,ablation_aggressive_nltk,track_b_transformer
0,My contactless tap isn't going through for $49...,my contactless tap isn't going through for _am...,contactless tap going even though entered pin,My contactless tap isn't going through for $49...
1,I have NOT received my £200 refund after 5 days.,i have not received my _amount_ refund after _...,received refund day,I have NOT received my £200 refund after 5 days.


,mode,description,vocab_size,val_accuracy,val_macro_f1,val_weighted_f1
0,raw_minimal,Raw Minimal (Unicode + whitespace only),9167,0.8793,0.8753,0.8793
1,classical_domain_aware,Domain-Aware Classical (Preserves negation + _...,9153,0.8786,0.8734,0.8786
2,classical_aggressive,Aggressive Classical (NLTK stopwords + WordNet...,5727,0.8559,0.8507,0.8557


## 4. Classical Baseline — TF-IDF (1–2 n-grams) + Logistic Regression
We tune `C` on the validation split and also record a `LinearSVC` reference to confirm Logistic Regression matches linear-SVM performance while providing native softmax probabilities for Top-3 routing.

In [4]:
baseline_artifact = tune_and_train_baseline(bundle)
display(pd.DataFrame(baseline_artifact['val_tuning_results']).round(4))
print(f"Selected best C = {baseline_artifact['best_c']}")
print('Reference LinearSVC Test Macro F1:', round(baseline_artifact['linear_svc_ref']['test']['macro_f1'], 4))

base_val_probs = predict_baseline_probs(baseline_artifact, bundle.val_texts)
base_test_probs = predict_baseline_probs(baseline_artifact, bundle.test_texts)
base_test_preds = np.argmax(base_test_probs, axis=1)
base_test_metrics = compute_classification_metrics(bundle.test_labels, base_test_preds)
base_lat = benchmark_single_query_latency(
    lambda q: predict_baseline_probs(baseline_artifact, [q]),
    bundle.test_texts[:50],
)
print('Baseline Test Metrics:', {k: round(v, 4) for k, v in base_test_metrics.items()})
print('Baseline Single-Query Latency (ms):', {k: round(v, 2) for k, v in base_lat.items()})

,C,val_accuracy,val_macro_f1,val_weighted_f1
0,0.5,0.8205,0.7997,0.8170
1,1.0,0.8499,0.8414,0.8484
2,2.0,0.8699,0.8651,0.8694
3,5.0,0.8786,0.8734,0.8786
4,10.0,0.8799,0.8764,0.8801


Selected best C = 10.0
Reference LinearSVC Test Macro F1: 0.8909
Baseline Test Metrics: {'accuracy': 0.8947, 'macro_precision': 0.8995, 'macro_recall': 0.8946, 'macro_f1': 0.8948, 'weighted_precision': 0.8995, 'weighted_recall': 0.8947, 'weighted_f1': 0.8949}
Baseline Single-Query Latency (ms): {'mean_ms': 0.79, 'median_ms': 0.89, 'p95_ms': 1.24}


## 5. Primary Model — Fine-Tuned DistilBERT (`distilbert-base-uncased`)
To guarantee fast `Restart & Run All` reproducibility on CPU, this cell loads saved fine-tuned weights from `models/transformer_intent/` when available (or runs training if `FORCE_RETRAIN_TRANSFORMER = True`).

In [5]:
if FORCE_RETRAIN_TRANSFORMER:
    print('FORCE_RETRAIN_TRANSFORMER=True -> Training DistilBERT end-to-end...')
    train_meta = train_distilbert(bundle)
    print('Training complete:', train_meta)

HAS_TRANSFORMER = transformer_weights_exist(TRANSFORMER_MODEL_DIR)
if HAS_TRANSFORMER:
    tr_model, tr_tokenizer, tr_device = load_transformer(TRANSFORMER_MODEL_DIR)
    tr_val_probs = predict_transformer_batch(
        bundle.val_texts, tr_model, tr_tokenizer, tr_device, label_names=bundle.label_names
    )
    tr_test_probs = predict_transformer_batch(
        bundle.test_texts, tr_model, tr_tokenizer, tr_device, label_names=bundle.label_names
    )
    tr_test_preds = np.argmax(tr_test_probs, axis=1)
    tr_test_metrics = compute_classification_metrics(bundle.test_labels, tr_test_preds)
    tr_lat = benchmark_single_query_latency(
        lambda q: predict_transformer_batch([q], tr_model, tr_tokenizer, tr_device, label_names=bundle.label_names, batch_size=1),
        bundle.test_texts[:30],
        n_warmup=3,
        n_runs=30,
    )
    print('DistilBERT loaded successfully on:', tr_device)
    print('DistilBERT Test Metrics:', {k: round(v, 4) for k, v in tr_test_metrics.items()})
    print('DistilBERT Single-Query Latency (ms):', {k: round(v, 2) for k, v in tr_lat.items()})
else:
    tr_val_probs = None
    tr_test_probs = None
    tr_test_preds = None
    print(
        '[NOTICE] Fine-tuned weights not yet in models/transformer_intent/. '
        'Run notebooks/colab_train_transformer.ipynb on Colab T4 GPU and extract weights here '
        'to unlock full DistilBERT comparison metrics and charts.'
    )

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

DistilBERT loaded successfully on: cpu
DistilBERT Test Metrics: {'accuracy': 0.9243, 'macro_precision': 0.9272, 'macro_recall': 0.9242, 'macro_f1': 0.9243, 'weighted_precision': 0.9273, 'weighted_recall': 0.9243, 'weighted_f1': 0.9244}
DistilBERT Single-Query Latency (ms): {'mean_ms': 18.65, 'median_ms': 18.31, 'p95_ms': 22.01}


## 6. Side-by-Side Evaluation & Focused Confusion Analysis
We compare models across Accuracy, Macro/Weighted Precision, Recall, F1, latency, and size on disk, then inspect the highest-error semantic clusters and top confused intent pairs.

In [6]:
comparison_rows = [
    {
        'model': 'TF-IDF + Logistic Regression (Baseline)',
        **base_test_metrics,
        'mean_latency_ms': base_lat['mean_ms'],
        'p95_latency_ms': base_lat['p95_ms'],
        'size_mb': get_path_size_mb(BASELINE_MODEL_PATH),
    }
]
if HAS_TRANSFORMER:
    comparison_rows.append(
        {
            'model': f'Fine-Tuned {TRANSFORMER_BASE_MODEL}',
            **tr_test_metrics,
            'mean_latency_ms': tr_lat['mean_ms'],
            'p95_latency_ms': tr_lat['p95_ms'],
            'size_mb': get_path_size_mb(TRANSFORMER_MODEL_DIR),
        }
    )

metrics_df = pd.DataFrame(comparison_rows)
export_metrics_tables(
    metrics_df,
    OUTPUTS_DIR / 'metrics_comparison.csv',
    OUTPUTS_DIR / 'metrics_comparison.md',
)
display(metrics_df.round(4))
plot_model_comparison(metrics_df, OUTPUTS_DIR / 'model_comparison_bar.png')

primary_preds = tr_test_preds if HAS_TRANSFORMER else base_test_preds
primary_name = 'Fine-Tuned DistilBERT' if HAS_TRANSFORMER else 'TF-IDF + Logistic Regression'
best_cls, worst_cls = get_per_class_f1_summary(bundle.test_labels, primary_preds, bundle.label_names, top_n=8)
print(f'Top 8 Best-Performing Intents ({primary_name}):')
display(best_cls.round(4))
print(f'Bottom 8 Hardest Intents ({primary_name}):')
display(worst_cls.round(4))

confused_df = get_top_confused_pairs(bundle.test_labels, primary_preds, bundle.label_names, top_k=15)
confused_df.to_csv(OUTPUTS_DIR / 'top_confused_pairs.csv', index=False)
display(confused_df)
plot_top_confused_pairs(confused_df, primary_name, OUTPUTS_DIR / 'top_confused_pairs_bar.png')

plot_focused_confusion_cluster(
    bundle.test_labels,
    primary_preds,
    bundle.label_names,
    OUTPUTS_DIR / 'top_confused_cluster_cm.png',
    baseline_preds=base_test_preds if HAS_TRANSFORMER else None,
    n_classes=10,
)

,model,accuracy,macro_precision,macro_recall,macro_f1,weighted_precision,weighted_recall,weighted_f1,mean_latency_ms,p95_latency_ms,size_mb
0,TF-IDF + Logistic Regression (Baseline),0.8947,0.8995,0.8946,0.8948,0.8995,0.8947,0.8949,0.7930,1.2415,5.7262
1,Fine-Tuned distilbert-base-uncased,0.9243,0.9272,0.9242,0.9243,0.9273,0.9243,0.9244,18.6489,22.0142,256.3402


Top 8 Best-Performing Intents (Fine-Tuned DistilBERT):


,intent,precision,recall,f1,support
0,apple_pay_or_google_pay,1.0000,1.0,1.0000,40
1,card_about_to_expire,1.0000,1.0,1.0000,40
2,passcode_forgotten,1.0000,1.0,1.0000,40
3,request_refund,1.0000,1.0,1.0000,40
4,terminate_account,1.0000,1.0,1.0000,40
5,verify_top_up,1.0000,1.0,1.0000,40
6,get_physical_card,0.9756,1.0,0.9877,40
7,verify_source_of_funds,0.9756,1.0,0.9877,40


Bottom 8 Hardest Intents (Fine-Tuned DistilBERT):


,intent,precision,recall,f1,support
0,balance_not_updated_after_bank_transfer,0.7895,0.7500,0.7692,40
1,pending_transfer,0.7692,0.7692,0.7692,39
2,topping_up_by_card,0.8205,0.8000,0.8101,40
3,declined_transfer,0.9655,0.7000,0.8116,40
4,verify_my_identity,0.8095,0.8500,0.8293,40
5,why_verify_identity,0.8649,0.8000,0.8312,40
6,transfer_timing,0.8293,0.8500,0.8395,40
7,declined_card_payment,0.7708,0.9250,0.8409,40


,true_intent,predicted_intent,pair_label,count
0,why_verify_identity,verify_my_identity,why_verify_identity -> verify_my_identity,7
1,card_arrival,card_delivery_estimate,card_arrival -> card_delivery_estimate,4
2,card_delivery_estimate,card_arrival,card_delivery_estimate -> card_arrival,4
3,pin_blocked,change_pin,pin_blocked -> change_pin,4
4,declined_transfer,declined_card_payment,declined_transfer -> declined_card_payment,4
5,transfer_into_account,topping_up_by_card,transfer_into_account -> topping_up_by_card,4
6,balance_not_updated_after_bank_transfer,transfer_timing,balance_not_updated_after_bank_transfer -> tra...,4
7,fiat_currency_support,exchange_via_app,fiat_currency_support -> exchange_via_app,3
8,wrong_amount_of_cash_received,declined_cash_withdrawal,wrong_amount_of_cash_received -> declined_cash...,3
9,top_up_reverted,top_up_failed,top_up_reverted -> top_up_failed,3


## 7. Real-World Constraints — Confidence Gating & Out-of-Scope Stress Test
In production banking chatbots, a wrong confident routing decision harms user trust. We calibrate the confidence threshold `tau` on validation predictions and test how well it escalates a 20-query probe suite of non-banking OOS, ambiguous, and multi-intent messages.

In [7]:
probes = get_oos_and_edge_probes()
probe_texts = [p['text'] for p in probes]
oos_only_texts = [p['text'] for p in probes if p['category'] == 'out_of_scope']

if HAS_TRANSFORMER:
    val_probs_active = tr_val_probs
    oos_probs_active = predict_transformer_batch(
        oos_only_texts, tr_model, tr_tokenizer, tr_device, label_names=bundle.label_names
    )
else:
    val_probs_active = base_val_probs
    oos_probs_active = predict_baseline_probs(baseline_artifact, oos_only_texts)

curve_df = evaluate_confidence_thresholds(bundle.val_labels, val_probs_active, oos_probs=oos_probs_active)
recommended_tau = select_recommended_threshold(curve_df, min_coverage=0.80)
display(curve_df.round(4))
print(f'Recommended validation threshold (>= 80% auto-coverage): tau = {recommended_tau:.2f}')
plot_confidence_coverage_curve(
    curve_df, recommended_tau, primary_name, OUTPUTS_DIR / 'confidence_coverage_curve.png'
)

service = IntentClassifierService()
probe_rows = []
for p in probes:
    res = service.predict(p['text'], top_k=3, threshold=recommended_tau, compare_baseline=False)['primary']
    probe_rows.append(
        {
            'category': p['category'],
            'query': p['text'],
            'predicted_intent': res.top_intent,
            'confidence': round(res.confidence, 3),
            'escalated': res.escalate,
        }
    )
display(pd.DataFrame(probe_rows))

,threshold,automation_coverage,escalation_rate,automated_accuracy,accepted_error_rate,oos_rejection_rate
0,0.30,0.9980,0.0020,0.9171,0.0827,0.2500
1,0.35,0.9947,0.0053,0.9188,0.0807,0.2500
2,0.40,0.9927,0.0073,0.9200,0.0794,0.2500
3,0.45,0.9893,0.0107,0.9225,0.0767,0.2500
4,0.50,0.9853,0.0147,0.9242,0.0747,0.2500
5,0.55,0.9807,0.0193,0.9259,0.0727,0.3333
6,0.60,0.9733,0.0267,0.9301,0.0680,0.4167
7,0.65,0.9686,0.0314,0.9332,0.0647,0.5000
8,0.70,0.9653,0.0347,0.9337,0.0640,0.5833
9,0.75,0.9586,0.0414,0.9381,0.0594,0.5833


Recommended validation threshold (>= 80% auto-coverage): tau = 0.95


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

,category,query,predicted_intent,confidence,escalated
0,out_of_scope,I want to book a flight to Tokyo next week.,card_delivery_estimate,0.669,True
1,out_of_scope,What is the weather forecast for Sydney tomorrow?,card_delivery_estimate,0.924,True
2,out_of_scope,Cancel my Amazon Prime subscription and packag...,apple_pay_or_google_pay,0.988,False
3,out_of_scope,Can you recommend a good Italian restaurant ne...,supported_cards_and_currencies,0.601,True
4,out_of_scope,Reset my Netflix password because I forgot it.,passcode_forgotten,0.997,False
5,out_of_scope,How do I bake sourdough bread at home?,request_refund,0.848,True
6,out_of_scope,Schedule a dentist appointment for Friday morn...,cancel_transfer,0.248,True
7,out_of_scope,Who won the World Cup final in 2022?,card_about_to_expire,0.577,True
8,out_of_scope,Write a Python script to sort a dictionary by ...,passcode_forgotten,0.546,True
9,out_of_scope,Reserve a two-bedroom hotel room in Melbourne.,pending_top_up,0.220,True


## 8. Concrete Error Analysis & Saved Report Artifacts
Inspect real test-set failures (both Transformer wins over TF-IDF and remaining Transformer errors) for the 1–2 page Summary Report and final live demo walkthrough.

In [8]:
err_df = extract_error_cases(
    bundle.test_texts,
    bundle.test_labels,
    base_test_preds,
    base_test_probs,
    bundle.label_names,
    trans_preds=tr_test_preds,
    trans_probs=tr_test_probs,
    max_cases=10,
)
err_df.to_csv(OUTPUTS_DIR / 'error_analysis_cases.csv', index=False)
display(err_df)

exported = sorted(p.name for p in OUTPUTS_DIR.iterdir() if p.is_file())
print('Exported report assets in outputs/:')
for name in exported:
    print(' -', name)

,case_type,text,true_intent,baseline_pred,baseline_conf,transformer_pred,transformer_conf,likely_cause
0,Transformer Win (Baseline Wrong),What is the best way to exchange currency?,exchange_via_app,exchange_charge,0.422,exchange_via_app,0.999,Fine-grained semantic overlap
1,Transformer Win (Baseline Wrong),Topping up my card is not working because I be...,pending_top_up,top_up_failed,0.647,pending_top_up,0.999,Negation / polarity nuance
2,Transformer Win (Baseline Wrong),Do you know what the restriction of the dispos...,disposable_card_limits,get_disposable_virtual_card,0.459,disposable_card_limits,0.999,Fine-grained semantic overlap
3,Transformer Win (Baseline Wrong),This is very frustrating. I can't get my card ...,declined_cash_withdrawal,card_not_working,0.284,declined_cash_withdrawal,0.999,Negation / polarity nuance
4,Transformer Win (Baseline Wrong),What's the refund process?,request_refund,Refund_not_showing_up,0.716,request_refund,0.999,Short / underspecified query
5,Transformer Error (Limitation),Can I exchange currencies?,fiat_currency_support,exchange_via_app,0.737,exchange_via_app,0.999,Short / underspecified query
6,Transformer Error (Limitation),What can I use a virtual disposable card for?,disposable_card_limits,get_disposable_virtual_card,0.953,get_disposable_virtual_card,0.999,Fine-grained semantic overlap
7,Transformer Error (Limitation),When I try and to buy something using my card ...,declined_transfer,declined_card_payment,0.555,declined_card_payment,0.999,Fine-grained semantic overlap
8,Transformer Error (Limitation),How do I top up my card?,transfer_into_account,topping_up_by_card,0.840,topping_up_by_card,0.999,Paraphrase / lexical gap
9,Transformer Error (Limitation),My card is just not working at this time.,virtual_card_not_working,card_not_working,0.387,card_not_working,0.999,Negation / polarity nuance


Exported report assets in outputs/:
 - class_distribution.png
 - confidence_coverage_curve.png
 - error_analysis_cases.csv
 - metrics_comparison.csv
 - metrics_comparison.md
 - model_comparison_bar.png
 - preprocessing_ablation.csv
 - top_confused_cluster_cm.png
 - top_confused_pairs.csv
 - top_confused_pairs_bar.png
